In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/MUJ-DS-23FE10CDS00276
!ls

In [ ]:
%%writefile config.yaml
# ============================================================
# Who does your writing sound like? - project configuration
# Change settings here, not in the code.
# ============================================================

project:
  name: "Who does your writing sound like?"
  random_seed: 42                 # makes every run give the same results

data:
  # Project Gutenberg plain-text URL ({id} = book number)
  source_url: "https://www.gutenberg.org/cache/epub/{id}/pg{id}.txt"
  raw_dir: "data/raw"             # downloaded books are saved here
  chunk_words: 500                # words per text chunk
  max_train_chunks_per_book: 40   # cap so long books don't dominate
  max_test_chunks_per_author: 40

  # 10 novelists: 3 books to train on, 1 unseen book to test on
  authors:
    austen:     {name: "Jane Austen",        train: [1342, 158, 161],    test: 105}
    dickens:    {name: "Charles Dickens",    train: [730, 1400, 786],    test: 98}
    twain:      {name: "Mark Twain",         train: [74, 86, 1837],      test: 76}
    doyle:      {name: "Arthur Conan Doyle", train: [244, 2097, 139],    test: 2852}
    hardy:      {name: "Thomas Hardy",       train: [110, 107, 153],     test: 143}
    bronte:     {name: "Charlotte Bronte",   train: [1260, 9182, 30486], test: 1028}
    eliot:      {name: "George Eliot",       train: [145, 550, 507],     test: 6688}
    conrad:     {name: "Joseph Conrad",      train: [5658, 2021, 219],   test: 974}
    wells:      {name: "H. G. Wells",        train: [35, 36, 5230],      test: 159}
    stevenson:  {name: "R. L. Stevenson",    train: [120, 421, 848],     test: 43}

features:
  name_min_count: 3               # a word must appear 3+ times to count as a name
  name_cap_ratio: 0.9             # ...and be capitalised 90%+ of the time

model:
  tfidf_ngram_range: [2, 4]       # character n-grams of length 2-4
  tfidf_max_features: 20000
  style_weight: 0.1               # ~1/sqrt(91): balances style vs TF-IDF blocks
  max_iter: 5000
  model_path: "models/final_model.joblib"
  results_dir: "results"

llm:
  provider: "groq"
  model: "openai/gpt-oss-120b"     # swap here if the provider retires this model
  api_key_env: "GROQ_API_KEY"     # name of the env variable, never the key itself
  timeout_seconds: 20             # give up on a slow answer after this long
  max_retries: 3

benchmark:
  n_passages: 40                  # test chunks sent to the LLM

app:
  max_words: 1500                 # longer input is cut to this
  warn_below_words: 150           # show a "short text" warning below this

In [ ]:
%%writefile .gitignore
data/raw/
models/
__pycache__/
.ipynb_checkpoints/
.env

In [ ]:
import yaml
cfg = yaml.safe_load(open('config.yaml'))
authors = cfg['data']['authors']
print(len(authors), "authors,", sum(len(a['train']) + 1 for a in authors.values()), "books")

!git add config.yaml .gitignore
!git commit -m "Add project config and gitignore"
!git push

In [ ]:
%%writefile src/__init__.py

In [ ]:
%%writefile src/utils.py
"""Small shared helpers."""
from pathlib import Path

import yaml


def load_config(path="config.yaml"):
    """Read the YAML config file into a dictionary."""
    with open(path, encoding="utf-8") as f:
        return yaml.safe_load(f)


def ensure_dir(path):
    """Create a folder (and parents) if it doesn't exist; return it as a Path."""
    path = Path(path)
    path.mkdir(parents=True, exist_ok=True)
    return path

In [ ]:
%%writefile src/data.py
"""Step 1 of the pipeline: download books, clean them, cut them into chunks,
and split into train/test (one whole unseen book per author for testing)."""
import re
import time
import urllib.request
from pathlib import Path

import pandas as pd

from src.features import normalize_punctuation
from src.utils import ensure_dir, load_config


def download_book(book_id, url_template, raw_dir):
    """Download one Gutenberg book (skips the download if already saved)."""
    path = Path(raw_dir) / f"{book_id}.txt"
    if not path.exists():
        url = url_template.format(id=book_id)
        with urllib.request.urlopen(url, timeout=60) as resp:
            path.write_bytes(resp.read())
        time.sleep(1)                      # be polite to Gutenberg's servers
    return path.read_text(encoding="utf-8", errors="ignore")


def get_title(raw_text):
    """Read the 'Title:' line from the Gutenberg header (used as a sanity check)."""
    match = re.search(r"^Title:\s*(.+)$", raw_text, flags=re.M)
    return match.group(1).strip() if match else "UNKNOWN"


def clean_text(raw_text):
    """Keep only the book itself and remove formatting that isn't the author's style."""
    start = re.search(r"\*\*\* ?START OF (THE|THIS) PROJECT GUTENBERG.*?\*\*\*", raw_text)
    end = re.search(r"\*\*\* ?END OF (THE|THIS) PROJECT GUTENBERG", raw_text)
    text = raw_text[start.end() if start else 0: end.start() if end else len(raw_text)]
    text = re.sub(r"\[Illustration[^\]]*\]", " ", text)                   # picture captions
    text = re.sub(r"^\s*(CHAPTER|Chapter|BOOK|Book|PART|Part)\b.*$", " ", text, flags=re.M)  # headings
    text = text.replace("_", "")                                            # _italics_ markers
    return normalize_punctuation(text)


def chunk_words(text, size):
    """Cut text into consecutive chunks of exactly `size` words (leftover dropped)."""
    words = text.split()
    return [" ".join(words[i:i + size]) for i in range(0, len(words) - size + 1, size)]


def build_dataset(cfg):
    """Download every book in the config and return one row per chunk."""
    d = cfg["data"]
    raw_dir = ensure_dir(d["raw_dir"])
    rows = []
    for author, info in d["authors"].items():
        books = [(b, "train") for b in info["train"]] + [(info["test"], "test")]
        for book_id, split in books:
            raw = download_book(book_id, d["source_url"], raw_dir)
            chunks = chunk_words(clean_text(raw), d["chunk_words"])
            print(f"{author:10s} {split:5s} {book_id:>6} {len(chunks):4d} chunks  {get_title(raw)[:45]}")
            rows += [{"text": c, "author": author, "book_id": book_id, "split": split, "position": i}
                     for i, c in enumerate(chunks)]
    return pd.DataFrame(rows)


def cap_chunks(df, cfg):
    """Randomly keep at most N chunks per train book and per author's test book."""
    d, seed = cfg["data"], cfg["project"]["random_seed"]
    parts = []
    for _, g in df[df.split == "train"].groupby("book_id"):
        parts.append(g.sample(min(len(g), d["max_train_chunks_per_book"]), random_state=seed))
    for _, g in df[df.split == "test"].groupby("author"):
        parts.append(g.sample(min(len(g), d["max_test_chunks_per_author"]), random_state=seed))
    return pd.concat(parts).reset_index(drop=True)


if __name__ == "__main__":
    cfg = load_config()
    df_all = build_dataset(cfg)
    df = cap_chunks(df_all, cfg)
    out = ensure_dir("data") / "chunks.csv"
    df.to_csv(out, index=False)
    print(f"\nSaved {len(df)} chunks to {out}\n")
    print(pd.crosstab(df["author"], df["split"], margins=True))

In [ ]:
!echo "data/chunks.csv" >> .gitignore
!python -m src.data

In [ ]:
!git add .gitignore src/__init__.py src/utils.py src/data.py
!git commit -m "Add data pipeline: download, clean, chunk, split Gutenberg books"
!git push

In [ ]:
%%writefile src/features.py
"""Step 2 of the pipeline: turn text into numbers that describe writing STYLE
(sentence length, punctuation, function words) and mask character names."""
import re
from collections import Counter

import nltk
import numpy as np
import pandas as pd
from nltk.tokenize import sent_tokenize

# NLTK's sentence splitter needs this model; download it once if missing
try:
    nltk.data.find("tokenizers/punkt_tab")
except LookupError:
    nltk.download("punkt_tab", quiet=True)

# Small, topic-independent words that authors use unconsciously
FUNCTION_WORDS = """the of and to a in that it is was he she i you for with as his her
but not be on at by this had have from or which they we all an were so my me
are their there what if would been one no will when who him them could some then
upon shall thus yet also very should may might must such own than only those these
whom where while though because nor""".split()

PUNCTUATION = {"comma": ",", "semicolon": ";", "colon": ":", "exclaim": "!",
               "question": "?", "dash": "--", "paren": "(", "quote": '"', "apostrophe": "'"}

WORD_RE = re.compile(r"[A-Za-z']+")

# Curly quotes / long dashes depend on who typed up the e-book, not on the author
PUNCT_MAP = str.maketrans({"\u201c": '"', "\u201d": '"', "\u2018": "'", "\u2019": "'",
                           "\u2014": "--", "\u2013": "--"})


def normalize_punctuation(text):
    """Turn typographic quotes/dashes into plain ASCII so every book is measured the same way."""
    return text.translate(PUNCT_MAP)


def style_features(text):
    """Return ~90 style measurements for one piece of text (rates are per 100 words)."""
    words = WORD_RE.findall(text.lower())
    n = max(len(words), 1)
    sent_lens = [len(s.split()) for s in sent_tokenize(text)] or [0]
    feats = {
        "avg_sent_len": float(np.mean(sent_lens)),
        "std_sent_len": float(np.std(sent_lens)),
        "avg_word_len": float(np.mean([len(w) for w in words])) if words else 0.0,
        "type_token_ratio": len(set(words)) / n,
    }
    for name, mark in PUNCTUATION.items():
        feats[f"p_{name}"] = text.count(mark) / n * 100
    counts = Counter(words)
    for w in FUNCTION_WORDS:
        feats[f"fw_{w}"] = counts[w] / n * 100
    return feats


def style_matrix(texts):
    """Apply style_features to many texts -> one row per text."""
    return pd.DataFrame([style_features(t) for t in texts])


def find_names(texts, min_count=3, cap_ratio=0.9):
    """Words that are (almost) always capitalised = probably names/places."""
    total, capital = Counter(), Counter()
    for t in texts:
        for w in re.findall(r"[A-Za-z]+", t):
            total[w.lower()] += 1
            capital[w.lower()] += w[0].isupper()
    return {w for w, c in total.items()
            if c >= min_count and len(w) > 1 and capital[w] / c > cap_ratio}


def mask_names(text, names):
    """Replace every name-like word with 'Xxx' so the model can't use them."""
    return re.sub(r"[A-Za-z]+", lambda m: "Xxx" if m.group().lower() in names else m.group(), text)


if __name__ == "__main__":
    from src.utils import load_config
    cfg = load_config()
    df = pd.read_csv("data/chunks.csv")
    X = style_matrix(df["text"])
    print("Style feature matrix:", X.shape)

    names = find_names(df.loc[df.split == "train", "text"],
                       cfg["features"]["name_min_count"], cfg["features"]["name_cap_ratio"])
    print(f"{len(names)} name-like words, e.g.", sorted(names)[:15])

    pd.set_option("display.width", 200)
    pd.set_option("display.max_columns", 20)
    cols = ["avg_sent_len", "avg_word_len", "p_semicolon", "p_dash", "p_exclaim", "fw_upon", "fw_very"]
    print(X[cols].groupby(df["author"]).mean().round(2))

In [ ]:
!python -m src.features

In [ ]:
!python -m src.data > /dev/null
!python -m src.features
!git add src/
!git commit -m "Add style features; normalise typographic punctuation"
!git push

In [ ]:
%%writefile src/train.py
"""Step 3 of the pipeline: train the authorship classifier, evaluate it on the
unseen test books, save results (metrics, confusion matrix) and the model."""
import json

import joblib
import matplotlib
matplotlib.use("Agg")                      # save plots to files (no screen needed)
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, classification_report, f1_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from src.features import find_names, mask_names, style_matrix
from src.utils import ensure_dir, load_config


def build_model(style_cols, cfg, use_tfidf=True):
    """Style features (scaled) [+ character n-gram TF-IDF] -> Logistic Regression."""
    m = cfg["model"]
    blocks = [("style", StandardScaler(), style_cols)]
    weights = {"style": 1.0}
    if use_tfidf:
        tfidf = TfidfVectorizer(analyzer="char_wb", ngram_range=tuple(m["tfidf_ngram_range"]),
                                sublinear_tf=True, min_df=2, max_features=m["tfidf_max_features"])
        blocks.append(("tfidf", tfidf, "text"))
        weights = {"style": m["style_weight"], "tfidf": 1.0}
    features = ColumnTransformer(blocks, transformer_weights=weights)
    return make_pipeline(features, LogisticRegression(max_iter=m["max_iter"]))


def make_inputs(texts, names):
    """Style features from the text + name-masked text for TF-IDF, in one table."""
    X = style_matrix(texts)
    X["text"] = [mask_names(t, names) for t in texts]
    return X


def main():
    cfg = load_config()
    out_dir = ensure_dir(cfg["model"]["results_dir"])
    df = pd.read_csv("data/chunks.csv")
    train, test = df[df.split == "train"], df[df.split == "test"]

    names = find_names(train["text"], cfg["features"]["name_min_count"], cfg["features"]["name_cap_ratio"])
    X_train, X_test = make_inputs(train["text"].tolist(), names), make_inputs(test["text"].tolist(), names)
    style_cols = [c for c in X_train.columns if c != "text"]

    # Compare: pure style vs style + TF-IDF
    results, models = {}, {}
    for label, use_tfidf in [("style_only", False), ("style_plus_tfidf", True)]:
        model = build_model(style_cols, cfg, use_tfidf).fit(X_train, train["author"])
        pred = model.predict(X_test)
        results[label] = {"accuracy": round(accuracy_score(test["author"], pred), 3),
                          "macro_f1": round(f1_score(test["author"], pred, average="macro"), 3)}
        models[label] = (model, pred)
        print(f"{label:18s} accuracy={results[label]['accuracy']:.3f}  macro-F1={results[label]['macro_f1']:.3f}")

    final_model, final_pred = models["style_plus_tfidf"]
    print("\n" + classification_report(test["author"], final_pred, digits=3))

    # Save metrics + confusion matrix
    (out_dir / "metrics.json").write_text(json.dumps(results, indent=2))
    fig, ax = plt.subplots(figsize=(8, 7))
    ConfusionMatrixDisplay.from_predictions(test["author"], final_pred, ax=ax, colorbar=False,
                                            xticks_rotation=45, cmap="Blues")
    ax.set_title("Confusion matrix - unseen test books")
    fig.tight_layout()
    fig.savefig(out_dir / "confusion_matrix.png", dpi=150)

    # Top style habits per author (what the model learned)
    lr = final_model[-1]
    coefs = pd.DataFrame(lr.coef_[:, :len(style_cols)], index=lr.classes_, columns=style_cols)
    lines = [f"{a:10s} " + ", ".join(coefs.loc[a].nlargest(5).index) for a in coefs.index]
    (out_dir / "top_style_features.txt").write_text("\n".join(lines))
    print("Top style features per author:\n" + "\n".join(lines))

    # Save everything the app needs in one file
    style_train = X_train[style_cols]
    bundle = {"model": final_model, "names": names, "style_cols": style_cols,
              "author_avg": style_train.groupby(train["author"].values).mean(),
              "overall_avg": style_train.mean()}
    ensure_dir("models")
    joblib.dump(bundle, cfg["model"]["model_path"])
    print(f"\nSaved model to {cfg['model']['model_path']} and results to {out_dir}/")


if __name__ == "__main__":
    main()

In [ ]:
!python -m src.train
!git add src/train.py results/
!git commit -m "Add training + evaluation; save metrics and confusion matrix"
!git push

In [ ]:
%%writefile prompts/prompts.yaml
# ============================================================
# LLM prompts. The app fills the {placeholders} in user_template
# with the classifier's results; the system prompt stays fixed.
# ============================================================

explainer:
  version: 1
  temperature: 0.4          # a little personality, but stays factual
  max_output_tokens: 2048
  system: |
    You are a friendly writing-style coach inside an app called
    "Who does your writing sound like?". A statistical model has already
    compared the user's text with 10 classic novelists. Your job is to explain
    its result to the user in plain, encouraging English.

    RULES
    1. Use ONLY the facts in the ANALYSIS block. Never invent facts about the
       author, their books, their life, or the user's text.
    2. Explain every habit in everyday words (e.g. "semicolons per 100 words"
       becomes "you link ideas with semicolons a lot").
    3. For each habit, compare the user's number with the author's number and
       the average of all authors.
    4. If the top match is below 35% or the text is under 150 words, say
       clearly that the match is tentative.
    5. Talk only about writing style, never about the topic or quality of the
       user's text.
    6. Maximum 120 words. No preamble, no closing remarks.

    OUTPUT FORMAT (Markdown)
    **You write most like <author>!** <one sentence on how confident the match is>
    - <habit 1, explained with the numbers>
    - <habit 2, explained with the numbers>
    - <habit 3, explained with the numbers>
    *Tip:* <one sentence on how to sound even more like this author, based only on the habits above>

    EXAMPLE
    ANALYSIS
    Closest author: Jane Austen
    Top matches: Jane Austen 48%, Charlotte Bronte 21%, George Eliot 12%
    Text length: 320 words
    Habits pushing toward Jane Austen:
    - uses of the word "very" per 100 words: you 0.90 | Jane Austen 0.49 | all authors 0.21
    - uses of the word "her" per 100 words: you 2.10 | Jane Austen 1.80 | all authors 1.10
    - average sentence length (words): you 27.5 | Jane Austen 28.1 | all authors 23.9

    ANSWER
    **You write most like Jane Austen!** It's a fairly clear match (48%).
    - You say "very" about 4x more than the average author, just like Austen (0.90 vs her 0.49).
    - You write a lot about "her": 2.10 per 100 words, close to Austen's 1.80 and well above the 1.10 average.
    - Your sentences run long and balanced, about 27 words, almost exactly Austen's 28.
    *Tip:* Keep your long, flowing sentences and let a well-placed "very" do the emphasis.

  user_template: |
    ANALYSIS
    Closest author: {author}
    Top matches: {top_matches}
    Text length: {n_words} words
    Habits pushing toward {author}:
    {habits}

    ANSWER

In [ ]:
%%writefile src/llm.py
"""LLM integration: sends the classifier's findings to an LLM (Groq API) and
returns a plain-English explanation. Prompts live in prompts/prompts.yaml."""
import os
import time

import yaml
from groq import Groq


def load_prompts(path="prompts/prompts.yaml"):
    """Read all prompt templates from the YAML prompt file."""
    with open(path, encoding="utf-8") as f:
        return yaml.safe_load(f)


class Explainer:
    """Wraps the LLM API call for the 'explainer' prompt."""

    def __init__(self, cfg, prompts):
        llm = cfg["llm"]
        self.model = llm["model"]
        self.retries = llm["max_retries"]
        self.prompt = prompts["explainer"]
        api_key = os.environ.get(llm["api_key_env"])
        # timeout: give up on a slow answer instead of freezing the app
        self.client = (Groq(api_key=api_key, timeout=llm["timeout_seconds"], max_retries=0)
                       if api_key else None)

    def explain(self, **facts):
        """Fill the prompt template with the analysis facts and ask the LLM.
        Returns the explanation text, or None if the LLM is unavailable."""
        if self.client is None:
            return None
        messages = [{"role": "system", "content": self.prompt["system"]},
                    {"role": "user", "content": self.prompt["user_template"].format(**facts)}]
        for attempt in range(self.retries):
            try:
                response = self.client.chat.completions.create(
                    model=self.model, messages=messages,
                    temperature=self.prompt["temperature"],
                    max_tokens=self.prompt["max_output_tokens"])
                text = response.choices[0].message.content
                if text:
                    return text.strip()
            except Exception as err:            # network/quota/timeout errors: wait, retry, then give up
                print(f"[LLM] call failed: {err}")
                time.sleep(2 ** attempt)        # wait 1s, 2s, 4s between tries
        return None

In [ ]:
!git add src/app.py src/llm.py
!git commit -m "Add Gradio demo app; retry Gemini with backoff"
!git push

In [ ]:
!git add config.yaml src/llm.py src/app.py
!git commit -m "Switch LLM to Groq (Llama 3.3 70B) with timeout and retries"
!git push

In [ ]:
%%writefile src/app.py
"""The demo: paste your writing -> closest classic author + an explanation
(the classifier finds the style habits, the LLM explains them)."""
import json
from pathlib import Path

import joblib
import pandas as pd

from src.features import mask_names, normalize_punctuation, style_features
from src.llm import Explainer, load_prompts
from src.utils import load_config

CFG = load_config()
BUNDLE = joblib.load(CFG["model"]["model_path"])
AUTHOR_NAMES = {k: v["name"] for k, v in CFG["data"]["authors"].items()}
EXPLAINER = Explainer(CFG, load_prompts())


def describe(feat):
    """Turn a feature name like 'fw_upon' into readable words."""
    if feat.startswith("fw_"):
        return f'uses of the word "{feat[3:]}" per 100 words'
    if feat.startswith("p_"):
        return f"{feat[2:]} marks per 100 words"
    return {"avg_sent_len": "average sentence length (words)",
            "std_sent_len": "variation in sentence length",
            "avg_word_len": "average word length (letters)"}.get(feat, feat)


def analyze(text):
    """Run the classifier on the text and collect the facts to explain."""
    words = normalize_punctuation(text).split()
    if not any(c.isalpha() for c in text):
        return None
    words = words[:CFG["app"]["max_words"]]
    clean = " ".join(words)

    feats = style_features(clean)
    X = pd.DataFrame([feats])[BUNDLE["style_cols"]]
    X["text"] = mask_names(clean, BUNDLE["names"])

    model = BUNDLE["model"]
    probs = dict(zip(model.classes_, model.predict_proba(X)[0]))
    best = max(probs, key=probs.get)

    # How much each style feature pushed toward the winner = weight x scaled value
    scaler = model[0].named_transformers_["style"]
    z = pd.Series(scaler.transform(X[BUNDLE["style_cols"]])[0], index=BUNDLE["style_cols"])
    coef = pd.Series(model[-1].coef_[list(model.classes_).index(best)][:len(z)], index=z.index)
    push = (coef * z).drop("type_token_ratio").nlargest(3)   # TTR is unreliable for short texts

    name = AUTHOR_NAMES[best]
    habits = [f"- {describe(f)}: you {feats[f]:.2f} | {name} {BUNDLE['author_avg'].loc[best, f]:.2f} "
              f"| all authors {BUNDLE['overall_avg'][f]:.2f}" for f in push.index]
    top3 = sorted(probs.items(), key=lambda kv: -kv[1])[:3]
    return {"author": name, "n_words": len(words), "habits": "\n".join(habits),
            "top_matches": ", ".join(f"{AUTHOR_NAMES[a]} {p:.0%}" for a, p in top3),
            "probs": {AUTHOR_NAMES[a]: float(p) for a, p in probs.items()}}


LEAF = ('<svg viewBox="0 0 24 24" width="{s}" height="{s}"><path d="M12 2C6.5 6 4.5 12 6.5 18'
        'c1.6 2.6 5.5 4 5.5 4s3.9-1.4 5.5-4C19.5 12 17.5 6 12 2z" fill="{c}"/></svg>')
SPINE_COLOURS = ["#9B3F6E", "#B4604F", "#C58A4A", "#7E3460", "#8E4A62",
                 "#6E2C55", "#A35A58", "#7A3B5C", "#94506F", "#6B2F57"]


def shelf_html(probs):
    """Draw every author as a book spine: the taller the spine, the closer the match."""
    ranked = sorted(probs.items(), key=lambda kv: -kv[1])
    top_name, top_p = ranked[0]
    spines = []
    for i, (name, p) in enumerate(ranked):
        height = 86 + 136 * (p / top_p)
        width = [56, 46, 42][i] if i < 3 else 30
        label = name if i == 0 else name.split()[-1]
        pct = f"<span class='pct'>{p:.0%}</span>" if i < 3 else ""
        ribbon = "<span class='ribbon'></span>" if i == 0 else ""
        cls = "spine top" if i == 0 else ("spine near" if i < 3 else "spine")
        spines.append(f"<div class='{cls}' style='height:{height:.0f}px;width:{width}px;"
                      f"background:{SPINE_COLOURS[i % len(SPINE_COLOURS)]}'>{ribbon}"
                      f"<span class='title'>{label}</span>{pct}</div>")
    strength = "a clear match" if top_p >= 0.45 else ("a fair match" if top_p >= 0.30 else "a tentative match")
    return (f"<div class='bookplate'><p class='ex'>Ex libris</p><p class='who'>{top_name}</p>"
            f"<p class='how'>{strength} · {top_p:.0%}</p></div>"
            f"<p class='label'>Taller spine · closer match</p>"
            f"<div class='shelf'>{''.join(spines)}</div><div class='plank'></div>")


EMPTY_SHELF = ("<div class='bookplate empty'><p class='ex'>Ex libris</p><p class='who'>· · ·</p>"
               "<p class='how'>your kindred author appears here</p></div>"
               "<p class='label'>Taller spine · closer match</p>"
               "<div class='shelf'>" + "".join(
                   f"<div class='spine ghost' style='height:{h}px;width:{w}px'></div>"
                   for h, w in [(150, 40), (120, 34), (170, 44), (100, 30), (135, 36), (90, 28), (115, 32)])
               + "</div><div class='plank'></div>")
EMPTY_WHY = "*Paste your writing and press **Find my author**.*"


def who_do_i_sound_like(text):
    """Gradio callback: returns (bookshelf HTML, explanation markdown)."""
    facts = analyze(text)
    if facts is None:
        return EMPTY_SHELF, "Please paste some text first."
    probs = facts.pop("probs")
    explanation = EXPLAINER.explain(**facts)
    if explanation is None:                  # LLM unavailable -> plain fallback, demo never breaks
        explanation = (f"**Style habits behind this match:**\n{facts['habits']}\n\n"
                       "_(AI explanation unavailable - showing the raw analysis.)_")
    explanation = explanation.replace("\n*Tip:", "\n\n*Tip:")   # keep the tip out of the bullet list
    if facts["n_words"] < CFG["app"]["warn_below_words"]:
        explanation += f"\n\n*Only {facts['n_words']} words - results get more reliable above ~200.*"
    return shelf_html(probs), explanation


def word_count_note(text):
    """Live word counter shown under the text box."""
    n = len(text.split())
    if n == 0:
        return "<p class='wc'>0 words</p>"
    if n < CFG["app"]["warn_below_words"]:
        return f"<p class='wc'>{n} words · a little more makes the reading surer</p>"
    return f"<p class='wc'>{n} words · a good long read</p>"


def how_it_works_html():
    """One-line model summary for the footer (reads the saved metrics)."""
    acc = ""
    metrics_file = Path(CFG["model"]["results_dir"]) / "metrics.json"
    if metrics_file.exists():
        m = json.loads(metrics_file.read_text())["style_plus_tfidf"]
        acc = f" · {m['accuracy']:.0%} right on books it has never seen"
    return (f"<div id='how'><span class='h'>How it works</span>"
            f"<span>91 style measurements · trained on 40 novels by {len(AUTHOR_NAMES)} authors{acc}"
            f" · explained by an LLM ({CFG['llm']['model']})</span></div>")


SAMPLE_CLASSIC = (
    "No one who had ever seen Catherine Morland in her infancy would have supposed her born to be an heroine. "
    "Her situation in life, the character of her father and mother, her own person and disposition, were all "
    "equally against her. Her father was a clergyman, without being neglected, or poor, and a very respectable man, "
    "though his name was Richard--and he had never been handsome. He had a considerable independence besides two "
    "good livings--and he was not in the least addicted to locking up his daughters. Her mother was a woman of "
    "useful plain sense, with a good temper, and, what is more remarkable, with a good constitution. She had three "
    "sons before Catherine was born; and instead of dying in bringing the latter into the world, as anybody might "
    "expect, she still lived on--lived to have six children more--to see them growing up around her, and to enjoy "
    "excellent health herself. A family of ten children will be always called a fine family, where there are heads "
    "and arms and legs enough for the number; but the Morlands had little other right to the word, for they were in "
    "general very plain, and Catherine, for many years of her life, as plain as any.")

SAMPLE_MODERN = (
    "I didn't plan to stay up this late, but here we are. The project was supposed to be simple: take some old "
    "books, teach a computer to recognise who wrote them, and build a little app around it. Simple, right? Then the "
    "model names changed, the API went down, and a dash turned out to be three different characters depending on "
    "who typed the book. Honestly, I think I learned more from the things that broke than from the things that "
    "worked. That's probably the real lesson here. You plan, you test, something odd happens, and you figure out "
    "why. Then you fix it and move on. My friends think I'm mad for caring about semicolons at midnight, and maybe "
    "they're right. But there's something satisfying about watching a program read a paragraph and say, with a "
    "straight face, that you write like a Victorian novelist. I'm not sure whether that's a compliment or an "
    "insult. Either way, I'm going to finish this, push it to GitHub, and sleep for about twelve hours.")

HERO = (
    "<div id='hero'>"
    + "".join(f"<span class='leaf' style='left:{x}%;animation-duration:{d}s;animation-delay:-{dl}s'>"
              + LEAF.format(s=sz, c=c) + "</span>"
              for x, d, dl, sz, c in [(6, 19, 3, 22, "#EFA07A"), (24, 23, 12, 18, "#D98C9A"),
                                      (61, 17, 7, 24, "#D8A657"), (83, 21, 15, 20, "#C8694C"),
                                      (93, 25, 1, 16, "#EFA07A")])
    + "<p class='kicker'>A stylometry reading room</p>"
      "<h1>Who does your writing <em>sound like?</em></h1>"
      "<div class='rule'><span></span>❦<span></span></div>"
      "<p class='sub'>Paste 200 words or more, and see which famous author you write like, "
      "and why.</p></div>")

CSS = """
@import url('https://fonts.googleapis.com/css2?family=Cormorant+Garamond:ital,wght@0,500;0,600;1,500&family=Lora:ital,wght@0,400;0,600;1,400&display=swap');

/* ---------- page ---------- */
body, gradio-app {
  background:
    radial-gradient(circle at 12% 55%, rgba(216,166,87,.30), transparent 32%),
    radial-gradient(circle at 88% 70%, rgba(239,160,122,.28), transparent 34%),
    radial-gradient(circle at 50% 40%, rgba(155,63,110,.45), transparent 30%),
    radial-gradient(ellipse 90% 60% at 50% 0%, #5B2340 0%, #3A1530 55%, #220C1D 100%) !important;
  background-attachment: fixed !important; background-color: #220C1D !important;
}
.gradio-container {
  --body-text-color: #F7E9D7 !important; --block-background-fill: transparent !important;
  --block-border-width: 0px !important; --block-shadow: none !important;
  --background-fill-primary: transparent !important; --background-fill-secondary: transparent !important;
  max-width: 1180px !important; margin: auto !important; position: relative;
  background: transparent !important; font-family: 'Lora', Georgia, serif !important; color: #F7E9D7 !important;
}
.gradio-container > * { position: relative; z-index: 1; }
footer { display: none !important; }

/* ---------- hero ---------- */
#hero { position: relative; text-align: center; padding: 46px 12px 30px; overflow: visible; }
#hero .kicker { margin: 0; font-family: 'Cormorant Garamond', serif; font-size: 15px; letter-spacing: .32em;
                text-transform: uppercase; color: #D8A657; }
#hero h1 { margin: 10px auto 0; max-width: 820px; font-family: 'Cormorant Garamond', serif; font-weight: 600;
           font-size: clamp(38px, 6vw, 62px); line-height: 1.05; color: #F7E9D7; }
#hero h1 em { font-weight: 500; color: #EFA07A; }
#hero .rule { display: flex; justify-content: center; align-items: center; gap: 14px; color: #D8A657; margin: 14px 0; }
#hero .rule span { display: block; width: 90px; height: 1px; background: rgba(216,166,87,.6); }
#hero .sub { margin: 0 auto; max-width: 560px; font-style: italic; font-size: 18px; color: #E3C9BC; }
@keyframes leaf-fall { 0% { transform: translate(0,-40px) rotate(0); opacity: 0 } 10% { opacity: .9 }
                       100% { transform: translate(80px, 1100px) rotate(540deg); opacity: 0 } }
#hero .leaf { position: absolute; top: 0; animation: leaf-fall linear infinite; pointer-events: none; }
@media (prefers-reduced-motion: reduce) { #hero .leaf { animation: none; opacity: .6 } }

/* ---------- glass panels ---------- */
.glass { background: rgba(255,238,220,.08) !important; border: 1px solid rgba(255,222,190,.22) !important;
         border-radius: 18px !important; padding: 26px 28px !important;
         backdrop-filter: blur(22px) saturate(140%); -webkit-backdrop-filter: blur(22px) saturate(140%);
         box-shadow: 0 20px 50px rgba(20,4,16,.45), inset 0 1px 0 rgba(255,240,225,.18) !important; }
.glass .block, .glass .form, .glass .wrap { background: transparent !important; border: none !important; box-shadow: none !important; }
.panel-head { display: flex; justify-content: space-between; align-items: baseline; gap: 12px; }
.panel-head h2 { margin: 0; font-family: 'Cormorant Garamond', serif; font-weight: 600; font-size: 30px; color: #F7E9D7; }
.panel-head span { font-family: 'Cormorant Garamond', serif; font-size: 14px; letter-spacing: .22em;
                   text-transform: uppercase; color: #D8A657; }

/* ---------- text box ---------- */
#manuscript textarea {
  background-color: rgba(30,8,24,.45) !important; color: #F7E9D7 !important;
  border: 1px solid rgba(255,222,190,.2) !important; border-radius: 12px !important;
  font-family: 'Lora', Georgia, serif !important; font-size: 16px !important; line-height: 28px !important;
  padding: 14px 18px !important;
  background-image: repeating-linear-gradient(180deg, transparent 0 27px, rgba(216,166,87,.13) 27px 28px) !important;
  background-attachment: local !important;
}
#manuscript textarea::placeholder { color: #A98B95 !important; }
#manuscript textarea:focus { outline: 2px solid #D8A657 !important; }
.wc { margin: 4px 2px 0; font-size: 14px; color: #D8A657; }

/* ---------- buttons ---------- */
.ghost-btn { background: transparent !important; color: #F3C3A6 !important; border: 1px solid rgba(239,160,122,.6) !important;
             border-radius: 10px !important; font-family: 'Lora', Georgia, serif !important; box-shadow: none !important; }
.ghost-btn:hover { background: rgba(239,160,122,.12) !important; }
#clear-btn { background: rgba(255,238,220,.08) !important; color: #E3C9BC !important;
             border: 1px solid rgba(255,222,190,.2) !important; border-radius: 10px !important; box-shadow: none !important; }
#find-btn { background: #D8A657 !important; color: #2A0F24 !important; border: none !important; border-radius: 10px !important;
            font-family: 'Cormorant Garamond', serif !important; font-weight: 600 !important; font-size: 21px !important;
            letter-spacing: .04em; box-shadow: 0 6px 20px rgba(216,166,87,.35) !important; }
#find-btn:hover { background: #E7BC72 !important; }

/* ---------- results: bookplate + shelf ---------- */
.inner-frame { border: 1px solid rgba(216,166,87,.45); border-radius: 12px; padding: 18px 20px 6px; }
.bookplate { margin: 6px auto 18px; width: fit-content; text-align: center; padding: 14px 36px 12px;
             border: 1px solid rgba(216,166,87,.6); border-radius: 50% / 24%;
             box-shadow: inset 0 0 0 5px rgba(30,8,24,.2), inset 0 0 0 6px rgba(239,160,122,.4); }
.bookplate .ex { margin: 0; font-family: 'Cormorant Garamond', serif; font-size: 13px; letter-spacing: .4em;
                 text-transform: uppercase; color: #D8A657; }
.bookplate .who { margin: 2px 0 0; font-family: 'Cormorant Garamond', serif; font-style: italic; font-size: 44px;
                  line-height: 1.1; color: #F0C987; }
.bookplate .how { margin: 4px 0 0; font-size: 14px; color: #E3C9BC; }
.bookplate.empty .who { color: rgba(240,201,135,.5); }
.label { margin: 0 0 8px; font-family: 'Cormorant Garamond', serif; font-size: 13px; letter-spacing: .3em;
         text-transform: uppercase; color: #D8A657; }
.shelf { display: flex; align-items: flex-end; justify-content: center; gap: 6px; height: 232px; padding: 0 6px; }
.spine { position: relative; box-sizing: border-box; border-radius: 4px 4px 2px 2px; padding: 8px 0;
         display: flex; flex-direction: column; align-items: center; justify-content: space-between;
         box-shadow: inset 3px 0 0 rgba(255,235,215,.12), inset -3px 0 0 rgba(20,4,16,.2);
         transform-origin: bottom; animation: rise .7s ease-out both; }
.spine .title { writing-mode: vertical-rl; transform: rotate(180deg); font-family: 'Cormorant Garamond', serif;
                font-weight: 600; font-size: 13px; color: #F3D9C9; white-space: nowrap; overflow: hidden; }
.spine.near .title { font-size: 15px; color: #FFE9D8; }
.spine.top { box-shadow: inset 4px 0 0 rgba(255,235,215,.16), inset -4px 0 0 rgba(20,4,16,.22), 0 0 26px rgba(216,166,87,.5); }
.spine.top .title { font-size: 19px; color: #FBE3B4; letter-spacing: .05em; }
.spine .pct { font-size: 11px; color: #FFE9D8; }
.spine.top .pct { font-size: 12px; color: #FBE3B4; }
.spine .ribbon { position: absolute; top: -6px; right: 10px; width: 9px; height: 34px; background: #EFA07A;
                 clip-path: polygon(0 0, 100% 0, 100% 100%, 50% 80%, 0 100%); }
.spine.ghost { background: rgba(255,238,220,.07); border: 1px dashed rgba(255,222,190,.2); animation: none; }
@keyframes rise { from { transform: scaleY(0); opacity: 0 } to { transform: scaleY(1); opacity: 1 } }
.plank { height: 12px; margin: 0 -4px 6px; border-radius: 3px; background: #8A5238;
         box-shadow: 0 6px 14px rgba(20,4,16,.45), inset 0 2px 0 rgba(255,225,190,.22); }

/* ---------- explanation ---------- */
#why { background: rgba(30,8,24,.38) !important; border: 1px solid rgba(255,222,190,.16) !important;
       border-radius: 12px !important; padding: 16px 20px !important; min-height: 90px; }
#why, #why p, #why li { color: #F7E9D7 !important; font-size: 15px; line-height: 1.65; }
#why strong { color: #F0C987 !important; }
#why em { color: #F3C3A6; }

/* ---------- footer ---------- */
#how { display: flex; flex-wrap: wrap; justify-content: space-between; gap: 6px 24px; align-items: center;
       border-top: 1px solid rgba(255,222,190,.22); border-bottom: 1px solid rgba(255,222,190,.22);
       padding: 14px 4px; margin: 8px 0 30px; font-size: 14px; color: #E3C9BC; }
#how .h { font-family: 'Cormorant Garamond', serif; font-size: 20px; color: #F0C987; }
"""


def build_app():
    import gradio as gr
    theme = gr.themes.Base(font=[gr.themes.GoogleFont("Lora"), "Georgia", "serif"], radius_size="lg")

    with gr.Blocks(theme=theme, css=CSS, title="Who does your writing sound like?") as demo:
        gr.HTML(HERO)
        with gr.Row(equal_height=False):
            with gr.Column(scale=1, elem_classes="glass"):
                gr.HTML("<div class='panel-head'><h2>Your page</h2><span>Page i</span></div>")
                text = gr.Textbox(lines=12, max_lines=22, show_label=False, elem_id="manuscript",
                                  placeholder="Paste 200 or more words: an essay, a story, a long letter...")
                count = gr.HTML(word_count_note(""))
                with gr.Row():
                    classic_btn = gr.Button("Read a classic sample", size="sm", elem_classes="ghost-btn")
                    modern_btn = gr.Button("Read a modern sample", size="sm", elem_classes="ghost-btn")
                with gr.Row():
                    clear_btn = gr.Button("Clear", elem_id="clear-btn", scale=1)
                    find_btn = gr.Button("Find my author", elem_id="find-btn", scale=2)

            with gr.Column(scale=1, elem_classes="glass"):
                gr.HTML("<div class='panel-head'><h2>You write like....</h2><span>Ex libris</span></div>")
                shelf = gr.HTML(EMPTY_SHELF, elem_classes="inner-frame")
                why = gr.Markdown(EMPTY_WHY, elem_id="why")

        gr.HTML(how_it_works_html())

        text.change(word_count_note, text, count)
        classic_btn.click(lambda: SAMPLE_CLASSIC, None, text)
        modern_btn.click(lambda: SAMPLE_MODERN, None, text)
        clear_btn.click(lambda: ("", EMPTY_SHELF, EMPTY_WHY), None, [text, shelf, why])
        find_btn.click(who_do_i_sound_like, text, [shelf, why])
    return demo


if __name__ == "__main__":
    build_app().launch()

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')
%cd /content/drive/MyDrive/MUJ-DS-23FE10CDS00276

import os
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

!pip install -q gradio groq
from src.app import build_app
build_app().launch(share=True)

In [ ]:
!echo ".gradio/" >> .gitignore
!git add .gitignore config.yaml src/app.py
!git commit -m "Use openai/gpt-oss-120b on Groq; update app label"
!git push

In [ ]:
%%writefile requirements.txt
pandas>=2.0
numpy>=1.24
scikit-learn>=1.3
nltk>=3.8
pyyaml>=6.0
joblib>=1.3
matplotlib>=3.7
gradio>=5.0
groq>=0.9

In [ ]:
%%writefile README.md
# Who Does Your Writing Sound Like? ✍️

Paste any piece of writing and find out which classic novelist your **writing style** is closest to, and *why*.
A classical NLP model does the stylometry; an LLM turns its findings into a plain-English explanation.

| | |
|---|---|
| **Name** | Mahi Sharma |
| **Registration No.** | 23FE10CDS00276 |
| **Branch** | B.Tech CSE (Data Science) |
| **Section** | D |
| **GitHub** | [mahi-sharmas](https://github.com/mahi-sharmas) |
| **Course** | Natural Language Processing – Project |


---

## How it works

```
Your text ──► style features ──► Logistic Regression ──► closest author + top 3 habits ──► LLM (Groq) ──► explanation
             (91 numbers +                                (e.g. "uses 'very' 2x more
              char n-grams)                                 than average")
```

1. **Data** (`src/data.py`): 40 public-domain novels from Project Gutenberg, 10 authors × 4 books.
   Books are cleaned (licence text, chapter headings, transcription quirks like curly quotes removed) and cut into 500-word chunks.
2. **Features** (`src/features.py`): 91 style measurements per chunk: sentence length, word length,
   vocabulary richness, punctuation rates, and frequencies of 78 *function words* ("the", "upon", "very"…),
   which authors use unconsciously regardless of topic. Character names are masked so the model can't cheat.
3. **Model** (`src/train.py`): Logistic Regression on the style features + character 2–4-gram TF-IDF.
4. **LLM explanation** (`src/llm.py`): the model's top 3 style habits are sent to an LLM via the Groq API,
   which explains them in friendly language using **only** the numbers it is given (no invented facts).
5. **App** (`src/app.py`): a Gradio web interface.

## Evaluation (unseen books)

For **every** author, the model trains on 3 books and is tested on a 4th book it has never seen
(120 train / 40 test chunks per author, 1,600 chunks total).

| Model | Accuracy | Macro-F1 |
|---|---|---|
| Random guess (10 authors) | 0.100 | – |
| Style features only | 0.743 | 0.742 |
| **Style + character TF-IDF** | **0.843** | **0.841** |

- Easiest: Doyle (recall 1.00), Wells, Brontë, Austen (F1 ≈ 0.90).
- Hardest: Dickens (F1 0.67) and Stevenson (recall 0.65); Stevenson's test book (*Jekyll and Hyde*) is horror,
  while his training books are adventure stories, so the genre shift hurts.
- Accuracy drops on short texts; 200+ words gives reliable results.

![Confusion matrix](results/confusion_matrix.png)

## LLM integration

| File | Role |
|---|---|
| `prompts/prompts.yaml` | **Prompt file**: system prompt (role, 6 rules, fixed output format, one worked example) + user template with `{placeholders}` |
| `config.yaml` | **Config file**: authors/books, chunk sizes, model settings, LLM provider/model/timeout/retries |
| `src/llm.py` | Fills the template with the classifier's facts and calls the LLM |

Design choices:
- **Grounded:** the LLM never sees your text, only the classifier's numbers, so it can't hallucinate facts.
- **Robust:** 20 s timeout, 3 retries with exponential backoff, and a plain-analysis fallback, so the app never breaks if the LLM is down.
- **Swappable:** provider and model live in `config.yaml`. (This project switched from Gemini to Groq by editing two files.)

## Repository structure

```
├── config.yaml            # all settings
├── prompts/prompts.yaml   # LLM prompt file
├── src/
│   ├── data.py            # download, clean, chunk, split
│   ├── features.py        # style features + name masking
│   ├── train.py           # train, evaluate, save model + results
│   ├── llm.py             # LLM (Groq) explainer
│   ├── app.py             # Gradio demo
│   └── utils.py
├── results/               # metrics.json, confusion_matrix.png, top_style_features.txt
├── notebooks/             # run_project.ipynb (Colab)
└── requirements.txt
```

## Installation & running

**Get a free LLM key:** sign up at [console.groq.com](https://console.groq.com) → API Keys → Create API Key.

### Option A: Google Colab (easiest)
Open `notebooks/run_project.ipynb` in Colab, add your key as a Colab Secret named `GROQ_API_KEY`, and run all cells.

### Option B: Local
```bash
git clone https://github.com/mahi-sharmas/MUJ-DS-23FE10CDS00276.git
cd MUJ-DS-23FE10CDS00276
pip install -r requirements.txt
export GROQ_API_KEY="your-key"      # Windows: set GROQ_API_KEY=your-key

python -m src.data     # download + prepare books (~1-2 min first time)
python -m src.train    # train + evaluate, saves model and results/
python -m src.app      # launch the app
```
Without a key, the app still works and shows the raw analysis instead of the AI explanation.

## Limitations
- Trained on 19th-century English novels; modern writing gets the *closest* match, not a true one.
- Short texts (< 150 words) give unreliable results.
- Name masking is a heuristic (capitalised words), so it occasionally masks ordinary words.

## Credits
Books from [Project Gutenberg](https://www.gutenberg.org) (public domain). LLM served by [Groq](https://groq.com).